# Create HK ITC Awards (Innovation and Technology Commission (Hong Kong), Catalonia)

Creates HK ITC awards from the Hong Kong ITF public registry exposed via the www.itf.gov.hk/datagovhk Socrata API.

**Prerequisites:** run `scripts/local/hk_itc_to_s3.py` to fetch + aggregate + upload first.

**Data source:** Socrata resource `s9xt-n979` on www.itf.gov.hk/datagovhk (RAISC = Registre d'Ajuts i Subvencions de Catalunya). Scraper filters to `entitat_oo_aa_o_departament_1 LIKE '%HK ITC%'` AND excludes anonymized private-person rows, then aggregates monthly payment rows by `Reference No.` → one row per grant. ~3,219 unique grants 2016-2026 from ~7,365 named-beneficiary payment rows.

**S3 location:** `s3a://openalex-ingest/awards/hk_itc/hk_itc_projects.parquet`

**HK ITC funder in OpenAlex:** funder_id 4320321920 · display_name "Innovation and Technology Commission (Hong Kong)" · ES.

**Schema notes:**
- `funder_award_id` = the HK ITC `Reference No.` (e.g. `ARD/301 or ITS/047/18FX`), URL-stable across years.
- HK ITC beneficiaries are **institutions** (universities, foundations, companies) — `ra_social_del_beneficiari`. There is no PI name in the source. `lead_investigator` is NULL across all rows; institutional `affiliation.name` carries the beneficiary.
- The '' (Foreign/Others) placeholder beneficiary appears in ~200 rows — NULLed in the notebook (not a real institution).
- `amount` = aggregated HKD sum of monthly payments per grant.
- Anonymized rows (`ra_social_del_beneficiari` = 'Persona física' or 'Benef. no publicable') are EXCLUDED at scrape time — these are individual student bursaries GDPR-redacted at the source.

provenance `hk_itc_itf`, priority 197.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hk_itc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hk_itc/hk_itc_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.hk_itc_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.hk_itc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hk_itc_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row. If 0, STOP — the funder is missing from `openalex.common.funder`.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321920;

## Step 2: Create HK ITC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hk_itc_awards
USING delta
AS
WITH
hk_itc_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321920  -- Innovation and Technology Commission (Hong Kong)
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'HKD' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.funder_scheme) RLIKE '(ITSP|PRP|MRP|TCFS)' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,
        g.funder_scheme as funder_scheme,
        'hk_itc_itf' as provenance,
        CASE WHEN TRY_CAST(g.year AS INT) IS NOT NULL
             THEN CAST(CONCAT(g.year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CASE WHEN TRY_CAST(CAST(NULL AS INT) AS INT) IS NOT NULL
             THEN CAST(CONCAT(CAST(NULL AS INT), '-12-31') AS DATE) ELSE NULL END as end_date,
        TRY_CAST(g.year AS INT) as start_year,
        TRY_CAST(CAST(NULL AS INT) AS INT) as end_year,
        -- HK ITC has no PI names — beneficiary is the institution. lead_investigator carries only the affiliation; given/family/orcid NULL.
        -- §6.4a: NULL the '' placeholder beneficiary (Foreign/Others — not a real institution).
        CASE
            WHEN g.institution_name IS NOT NULL
             AND g.institution_name != ''
             AND TRIM(g.institution_name) != '' THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution_name as name,
                        'Hong Kong' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        'https://www.itf.gov.hk/datagovhk/Economia/Concessions-del-RAISC-Registre-de-subvencions-i-aju/s9xt-n979' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.hk_itc_raw g
    CROSS JOIN hk_itc_funder f
    WHERE g.funder_award_id IS NOT NULL
      AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hk_itc_itf' AND priority = 197;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    197 as priority
FROM openalex.awards.hk_itc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_hk_itc_awards FROM openalex.awards.hk_itc_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, amount, currency, start_year, lead_investigator.affiliation.name
FROM openalex.awards.hk_itc_awards LIMIT 10;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.hk_itc_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
-- §6.3 completeness. NOTE: pct_with_pi expected 0% — HK ITC's source has no PI names; beneficiary is the institution and lead_investigator carries only affiliation.name. pct_with_amount expected 100%. pct_with_institution expected ~94% (the '' Foreign/Others placeholder is NULLed, ~6% of rows).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.hk_itc_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.hk_itc_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.hk_itc_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name ORDER BY grant_count DESC LIMIT 20;